# Train LightGCN

In [ ]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "scipy<1.12" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# For SciPy 1.12+ compatibility: dok_matrix._update was removed
import scipy.sparse as sp
if not hasattr(sp.dok_matrix, '_update'):
    sp.dok_matrix._update = sp.dok_matrix.update

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [10]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty"
DATA_DIR: str = "../data"
SEED = 67

# if torch.cuda.is_available():
#     DEVICE = "cuda"
# elif torch.backends.mps.is_available():
#     DEVICE = "mps"
# else:
#     DEVICE = "cpu"

DEVICE = "cpu"

print(f"Using device: {DEVICE}")

Using device: cpu


## Create dataset

In [14]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "cold"],
    },
    "epochs": 10,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "n_layers": 3,
    "reg_weight": 1e-4,
    "seed": SEED,
}

config: Config = Config(model="LightGCN", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [15]:
dataset = create_dataset(config)


def _normalize_cold_feature(feat: pd.DataFrame, field: str = "cold") -> None:
    """Recover the literal 0/1 warm/cold labels for `feat[field]`.

    The `cold` column is declared as a TOKEN field, so RecBole remaps the
    binary labels onto a shared token vocabulary (e.g.
    ``{'[PAD]': 0, '0': 1, '1': 2}``) and applies that remap inconsistently
    across the user/item feats: one keeps the raw '0'/'1' strings while the
    other ends up with remapped ids. Both forms break here — the raw strings
    can't be cast to a LongTensor in ``data_preparation``, and the remapped
    ids no longer match the literal 0.0/1.0 the warm/cold eval compares
    against. Map every value back to its original label via the vocabulary.
    """
    token_of_id = {i: t for t, i in dataset.field2token_id[field].items()}

    def to_label(v: object) -> int:
        token = v if isinstance(v, str) else token_of_id.get(int(v), str(v))
        return 0 if token == "[PAD]" else int(token)

    feat[field] = feat[field].map(to_label).astype("int64")


_normalize_cold_feature(dataset.user_feat)
_normalize_cold_feature(dataset.item_feat)

train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
23 Jun 18:12    INFO  [Training]: train_batch_size = [1024] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
23 Jun 18:12    INFO  [Evaluation]: eval_batch_size = [409600000] 

## Train LightGCN

In [16]:
model: LightGCN = LightGCN(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
23 Jun 18:23    INFO  epoch 0 training [time: 673.74s, train loss: 425.9607]
23 Jun 18:23    INFO  epoch 0 evaluating [time: 13.24s, valid_score: 0.006500]
23 Jun 18:23    INFO  valid result: 
ndcg@20 : 0.0065    recall@20 : 0.0119    mrr@20 : 0.0077
23 Jun 18:23    INFO  Saving current: saved/LightGCN-Jun-23-2026_18-12-06.pth
23 Jun 18:34    INFO  epoch 1 training [time: 673.55s, train loss: 294.3923]
23 Jun 18:35    INFO  epoch 1 evaluating [time: 13.35s, valid_score: 0.006600]
23 Jun 18:35    INFO  valid result: 
ndcg@20 : 0.0066    recall@20 : 0.0121    mrr@20 : 0.0079
23 Jun 18:35    INFO  Saving current: saved/Ligh

In [17]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0077
Best valid result:
  ndcg@20: 0.0077
  recall@20: 0.0144
  mrr@20: 0.0088


## Evaluate on test set

In [18]:
test_result: dict[str, float] = trainer.evaluate(test_data, load_best_model=False)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

Test results (Overall):
  ndcg@20: 0.0074
  recall@20: 0.0133
  mrr@20: 0.0089


In [19]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)


/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue o

,Segment,Interactions,ndcg@20,recall@20,mrr@20
6,user-cold×item-warm,72341,0.0110,0.0210,0.0115
2,item-warm,88077,0.0109,0.0208,0.0117
4,user-warm×item-warm,15736,0.0103,0.0197,0.0131
1,user-cold,150964,0.0075,0.0135,0.0088
0,user-warm,46742,0.0065,0.0116,0.0097
3,item-cold,109629,0.0000,0.0000,0.0000
5,user-warm×item-cold,31006,0.0000,0.0000,0.0000
7,user-cold×item-cold,78623,0.0000,0.0000,0.0000
